In [2]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

model = "claude-sonnet-4-6"

In [3]:
tools = [
    {
        "name": "get_customer",
        "description": "Get customer profile by email.",
        "input_schema": {
            "type": "object",
            "properties": {
                "email": {"type": "string"}
            },
            "required": ["email"]
        }
    },
    {
        "name": "lookup_order",
        "description": "Look up an order by order id.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"}
            },
            "required": ["order_id"]
        }
    },
    {
        "name": "process_refund",
        "description": "Process a refund for an eligible order.",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "reason": {"type": "string"}
            },
            "required": ["order_id", "reason"]
        }
    },
    {
        "name": "escalate_to_human",
        "description": "Escalate the case to a human support agent.",
        "input_schema": {
            "type": "object",
            "properties": {
                "reason": {"type": "string"}
            },
            "required": ["reason"]
        }
    }
]

In [4]:
def get_customer(email):
    return {
        "customer_id": "cus_123",
        "email": email,
        "name": "Alex Johnson",
        "verified": True
    }


def lookup_order(order_id):
    return {
        "order_id": order_id,
        "status": "delivered",
        "item": "Wireless Keyboard",
        "refund_eligible": True
    }


def process_refund(order_id, reason):
    return {
        "order_id": order_id,
        "refund_status": "approved",
        "reason": reason
    }


def escalate_to_human(reason):
    return {
        "status": "escalated",
        "reason": reason
    }


tool_functions = {
    "get_customer": get_customer,
    "lookup_order": lookup_order,
    "process_refund": process_refund,
    "escalate_to_human": escalate_to_human
}

In [6]:
messages = [
    {
        "role": "user",
        "content": (
            "Hi, my email is alex@example.com. "
            "I want a refund for order ORD-1001 because the keyboard arrived damaged."
        )
    }
]


while True:
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        tools=tools,
        messages=messages
    )

    messages.append({
        "role": "assistant",
        "content": response.content
    })

    if response.stop_reason == "end_turn":
        final_text = response.content[0].text
        print(final_text)
        break

    if response.stop_reason == "tool_use":
        tool_results = []

        for block in response.content:
            if block.type == "tool_use":
                tool_name = block.name
                tool_input = block.input
                tool_use_id = block.id

                print(f"Claude requested tool: {tool_name}")
                print(f"Tool input: {tool_input}")

                result = tool_functions[tool_name](**tool_input)

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": tool_use_id,
                    "content": str(result)
                })

        messages.append({
            "role": "user",
            "content": tool_results
        })

        continue

    raise RuntimeError(f"Unexpected stop_reason: {response.stop_reason}")

Claude requested tool: get_customer
Tool input: {'email': 'alex@example.com'}
Claude requested tool: lookup_order
Tool input: {'order_id': 'ORD-1001'}
Claude requested tool: process_refund
Tool input: {'order_id': 'ORD-1001', 'reason': 'Keyboard arrived damaged'}
Your refund has been **approved**! 🎉 Here's a summary:

- **Order:** ORD-1001 — Wireless Keyboard
- **Reason:** Keyboard arrived damaged
- **Refund Status:** Approved ✅

You should see the refund reflected back to your original payment method within a few business days. If you have any other questions or concerns, feel free to ask!
